# Cadrille — Google Colab

Supports:
- **Eval**: run inference + metrics on a checkpoint (free T4 / Pro A100)
- **SFT smoke test**: 50-step training run to verify the pipeline (Pro A100 recommended)
- **Monitoring**: sync W&B offline runs

> Mount Google Drive first if your data / checkpoints live there.

In [ ]:
# ── [1] Mount Drive (optional — skip if using HuggingFace Hub) ──────────────
from google.colab import drive
drive.mount('/content/drive')

# Adjust these paths to match your Drive layout
DRIVE_DATA  = '/content/drive/MyDrive/cadrille/data'
DRIVE_CKPTS = '/content/drive/MyDrive/cadrille/checkpoints'

In [ ]:
# ── [2] Clone repo & install dependencies ───────────────────────────────────
import os

if not os.path.exists('/content/cadrille'):
    !git clone https://github.com/miachen0401/cadrille.git /content/cadrille

%cd /content/cadrille

!pip install -q \
    transformers>=4.50 accelerate qwen-vl-utils \
    flash-attn --no-build-isolation \
    cadquery trimesh open3d scipy wandb tqdm pyyaml

In [ ]:
# ── [3] Symlink data & checkpoints from Drive ────────────────────────────────
import os

if not os.path.exists('/content/cadrille/data'):
    os.symlink(DRIVE_DATA, '/content/cadrille/data')

if not os.path.exists('/content/cadrille/checkpoints'):
    os.symlink(DRIVE_CKPTS, '/content/cadrille/checkpoints')

!ls data/ && ls checkpoints/ | head -5

In [ ]:
# ── [4] GPU check ────────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory // 1024**3} GB')

In [ ]:
# ── [5a] EVAL — run inference on a checkpoint ─────────────────────────────────
# Edit CHECKPOINT_PATH and EVAL_SPLIT as needed.

CHECKPOINT_PATH = 'maksimko123/cadrille'          # HuggingFace Hub or local path
EVAL_SPLIT      = 'deepcad_test_mini'              # subdirectory under data/
MODE            = 'pc'                             # 'pc' | 'img'
PY_PATH         = '/tmp/cadrille_eval_preds'

import os
os.makedirs(PY_PATH, exist_ok=True)
assert len(os.listdir(PY_PATH)) == 0, f'{PY_PATH} must be empty — delete previous predictions first'

!python test.py \
    --checkpoint-path {CHECKPOINT_PATH} \
    --data-path ./data \
    --split {EVAL_SPLIT} \
    --mode {MODE} \
    --py-path {PY_PATH}

In [ ]:
# ── [5b] EVAL — compute IoU & Chamfer distance ────────────────────────────────
!python evaluate.py \
    --gt-mesh-path ./data/{EVAL_SPLIT} \
    --pred-py-path {PY_PATH}

In [ ]:
# ── [6] SFT smoke test (50 steps — verifies pipeline, needs Pro A100) ─────────
import wandb
wandb.login()   # paste your W&B API key when prompted

!python train.py \
    --config configs/sft-s600-lr3e-4-b2a4-pc_img.yaml \
    --max-steps 50 \
    --run-name colab-smoke-test \
    --wandb-project cadrille-sft

In [ ]:
# ── [7] Sync a stalled W&B run (after rate-limit crash) ──────────────────────
# Point to the run directory on Drive.

WANDB_RUN_DIR = f'{DRIVE_CKPTS}/../wandb/run-XXXXXXXX-RUNID'

!python -m wandb sync {WANDB_RUN_DIR}